# Demonstrating the Implementation of Issue #424: Async LogScanner Iteration

In this notebook, we will demonstrate the new Python API ergonomics for reading from an Apache Fluss `LogScanner`.
Previously, achieving record consumption required manually polling batches (`scanner.poll()`), checking timeouts, extracting iterators from internal structures (`ScanRecords`), and unwieldy while-loops.

On our new PyO3 Rust branch `feat/424-python-async-iterator`, we have implemented the Python standard `__aiter__` and `__anext__` dunder methods on the `LogScanner`.

### Under the Hood
Because the Python `await` unblocks the internal event loop, the Rust boundary cannot simply hold a borrowed `&mut self` pointing back to the PyO3 class. To solve this, we wrapped the internal state inside a `Arc<tokio::sync::Mutex<ScannerState>>`.

This notebook gives a true "Before vs. After" side-by-side comparison using a live local server!

### Step 1: Connecting to the Cluster
Make sure your Fluss cluster is running locally. (In another terminal, ensure the server is bound to localhost).

In [1]:
import sys

# 1. Install maturin specifically into the Notebook's Python environment
!{sys.executable} -m pip install maturin

# 2. Compile and install your local fluss-rust code into this environment
!cd /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python && {sys.executable} -m maturin develop

  Using cached maturin-1.12.6-py3-none-macosx_10_12_x86_64.whl.metadata (16 kB)
Using cached maturin-1.12.6-py3-none-macosx_10_12_x86_64.whl (9.8 MB)

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
🍹 Building a mixed python/rust project
🔗 Found pyo3 bindings
🐍 Found CPython 3.12 at /Users/jaredyu/Desktop/open_source/fluss-notes/.venv/bin/python
📡 Using build options features from pyproject.toml
Ignoring mypy: markers 'extra == "dev"' don't match your environment
Ignoring pytest: markers 'extra == "dev"' don't match your environment
Ignoring pytest-asyncio: markers 'extra == "dev"' don't match your environment
Ignoring pytest-xdist: markers 'extra == "dev"' don't match your environment
Ignoring ruff: markers 'extra == "dev"' don't match your environment
Ignoring maturin: markers 'extra == "dev"' don't match your environment
Ignoring testcontainers: markers 'extra == "dev"' don't match your environment
Ignoring pdoc: markers 

In [2]:
import fluss
import os
from datetime import datetime

# Get the exact path of the loaded compiled _fluss extension
extension_path = fluss._fluss.__file__
mod_time = os.path.getmtime(extension_path)

print(f"Loaded Extension: {extension_path}")
print(f"Compiled At: {datetime.fromtimestamp(mod_time)}")

ModuleNotFoundError: No module named 'fluss'

In [15]:
import fluss

print("Is __aiter__ present?", hasattr(fluss.LogScanner, "__aiter__"))
print("All native Rust methods:", [m for m in dir(fluss.LogScanner) if not m.startswith("_") or m in ("__aiter__", "__anext__")])

Is __aiter__ present? True
All native Rust methods: ['__aiter__', 'create_empty_table', 'poll', 'poll_arrow', 'poll_record_batch', 'subscribe', 'subscribe_buckets', 'subscribe_partition', 'subscribe_partition_buckets', 'to_arrow', 'to_pandas', 'unsubscribe', 'unsubscribe_partition']


In [16]:
import fluss
print(fluss.__file__)

/Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python/fluss/__init__.py


In [17]:
import asyncio
import traceback
import pyarrow as pa

# This automatically grabs the native Rust-compiled wheel you built via `maturin develop`
import fluss

SERVER_URL = "localhost:9123" # Or adjust to your local port

async def setup_cluster():
    conf = fluss.Config()
    conf.bootstrap_servers = SERVER_URL
    
    connection = await fluss.FlussConnection.create(conf)
    admin = await connection.get_admin()
    
    print("Successfully connected to the admin endpoint!")
    return connection, admin

connection, admin = await setup_cluster()

Successfully connected to the admin endpoint!


In [18]:
table_path = fluss.TablePath("fluss", "async_iterator_demo")

async def provision_table():
    if await admin.table_exists(table_path):
        await admin.drop_table(table_path, ignore_if_not_exists=True)
        
    schema = fluss.Schema(
        pa.schema([
            pa.field("id", pa.int32()),
            pa.field("event_name", pa.string()),
            pa.field("value", pa.float64())
        ])
    )
    
    descriptor = fluss.TableDescriptor(schema)
    await admin.create_table(table_path, descriptor, ignore_if_exists=False)
    print(f"Provisioned '{table_path}' successfully.")

    table = await connection.get_table(table_path)
    writer = table.new_append().create_writer()
    
    print("Publishing records...")
    for i in range(10):
        writer.append({"id": i, "event_name": f"user_click_{i}", "value": i * 2.5})
    await writer.flush()
    print("Records generated!")

await provision_table()

Provisioned 'fluss.async_iterator_demo' successfully.
Publishing records...
Records generated!


### Step 2: The Old Way (Before Issue #424)
This was the legacy looping mechanism. It feels very Java-like and demands timeout injection, explicit extraction, and breaks pythonic semantics.

In [19]:
async def demo_legacy_polling():
    table = await connection.get_table(table_path)
    scanner = await table.new_scan().create_log_scanner()
    
    num_buckets = (await admin.get_table_info(table_path)).num_buckets
    scanner.subscribe_buckets({i: fluss.EARLIEST_OFFSET for i in range(num_buckets)})
    
    print("----- Legacy Reading Start -----")
    # Note we aren't `await`ing anything besides custom loops
    polled_count = 0
    for _ in range(3): # Poll loop
        scan_records = scanner.poll(2000) # blocking 2 second timeout 
        
        for record in scan_records:
            print(f"Polled Row: {record.row}")
            polled_count += 1
            
        if polled_count >= 10:
            break

    print("----- Legacy Reading Finish -----")
    
await demo_legacy_polling()

----- Legacy Reading Start -----
Polled Row: {'id': 0, 'event_name': 'user_click_0', 'value': 0.0}
Polled Row: {'id': 1, 'event_name': 'user_click_1', 'value': 2.5}
Polled Row: {'id': 2, 'event_name': 'user_click_2', 'value': 5.0}
Polled Row: {'id': 3, 'event_name': 'user_click_3', 'value': 7.5}
Polled Row: {'id': 4, 'event_name': 'user_click_4', 'value': 10.0}
Polled Row: {'id': 5, 'event_name': 'user_click_5', 'value': 12.5}
Polled Row: {'id': 6, 'event_name': 'user_click_6', 'value': 15.0}
Polled Row: {'id': 7, 'event_name': 'user_click_7', 'value': 17.5}
Polled Row: {'id': 8, 'event_name': 'user_click_8', 'value': 20.0}
Polled Row: {'id': 9, 'event_name': 'user_click_9', 'value': 22.5}
----- Legacy Reading Finish -----


### Step 3: The New Pythonic Way (After Issue #424)
Now we can just use `async for`! The stream will cleanly yield items back into the Python EventLoop one by one until the backend detects closure. No manual timeout parsing, no nested loops.

In [20]:
async def demo_async_for_loop():
    table = await connection.get_table(table_path)
    scanner = await table.new_scan().create_log_scanner()
    # scanner = await table.new_scan().create_record_batch_log_scanner()
    
    num_buckets = (await admin.get_table_info(table_path)).num_buckets
    scanner.subscribe_buckets({i: fluss.EARLIEST_OFFSET for i in range(num_buckets)})
    
    print("----- Async Iteration Reading Start -----")
    
    count = 0
    try:
        # Beautiful and idiomatic!
        async for record in scanner:
            print(f"Yielded Row asynchronously: {record.row}")
            count += 1
            
            if count >= 10:
                break
                
    except Exception as e:
        print(f"Stream complete or error triggered: {e}")
        traceback.print_exc()

    print("----- Async Iteration Reading Finish -----")

await demo_async_for_loop()

----- Async Iteration Reading Start -----
Yielded Row asynchronously: {'id': 0, 'event_name': 'user_click_0', 'value': 0.0}
Yielded Row asynchronously: {'id': 1, 'event_name': 'user_click_1', 'value': 2.5}
Yielded Row asynchronously: {'id': 2, 'event_name': 'user_click_2', 'value': 5.0}
Yielded Row asynchronously: {'id': 3, 'event_name': 'user_click_3', 'value': 7.5}
Yielded Row asynchronously: {'id': 4, 'event_name': 'user_click_4', 'value': 10.0}
Yielded Row asynchronously: {'id': 5, 'event_name': 'user_click_5', 'value': 12.5}
Yielded Row asynchronously: {'id': 6, 'event_name': 'user_click_6', 'value': 15.0}
Yielded Row asynchronously: {'id': 7, 'event_name': 'user_click_7', 'value': 17.5}
Yielded Row asynchronously: {'id': 8, 'event_name': 'user_click_8', 'value': 20.0}
Yielded Row asynchronously: {'id': 9, 'event_name': 'user_click_9', 'value': 22.5}
----- Async Iteration Reading Finish -----


### Step 4: Compiling and Testing the Core Locally
If you want to compile these changes locally and push them to Github, here is the official workflow as documented in `DEVELOPMENT.md`:

```bash
# 1. Initialize UV bindings 
cd bindings/python
uv sync --all-extras
source .venv/bin/activate

# 2. Re-compile the PyO3 Rust extension specifically into your local virtual environment
maturin develop

# 3. Run the automated integration test matrix
# Ensure you have Docker running locally as `testcontainers` invokes `docker run`
pytest test/test_log_table.py -k "test_async_iterator" -v
```
